# GTEx CLAMP models with all pathways prior (all samples)

**Environment:** `clamp-analyses`

Runs CLAMPfull with all pathways prior (Hallmark, Reactome, GO CC, C8) using all GTEx samples, reusing the FBM, SVD, and CLAMPbase results already generated in `nbs/01_model_building/02_gtex/01_CLAMP.ipynb`.

Steps:
1. Load existing FBM, SVD, CLAMPbase, and CLAMP_K from `config$GTEx$OUTPUT_DIR`
2. Run CLAMPfull with the combined all-pathways prior
3. Save results to `config$GTEx$OUTPUT_DIR/CLAMPfull_hall`

## Load libraries

In [1]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(Matrix)
library(here)
library(CLAMP)

source(here("config.R"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Configuration

In [2]:
output_data_dir  <- config$GTEx$OUTPUT_DIR
output_write_dir <- config$GTEx$CLAMP_HALL_DIR
pathways_path   <- here::here('data/pathways')

MULTIPLIER <- 100
MAX_ITER   <- 5000

message("GTEx output dir: ", output_data_dir)

GTEx output dir: /home/msubirana/Documents/pivlab/clamp-analyses/output/gtex



## Load pre-built GTEx inputs

In [3]:
gtex_genes    <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))
samples       <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))
gtex_svdRes   <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))
gtex_baseRes  <- readRDS(file.path(output_data_dir, "CLAMPbase.rds"))
CLAMP_K_gtex  <- readRDS(file.path(output_data_dir, "CLAMP_K_gtex.rds"))

message("Genes: ", length(gtex_genes))
message("Samples: ", length(samples))
message("CLAMP K: ", CLAMP_K_gtex)

Genes: 21613

Samples: 17382

CLAMP K: 578



## Load and match all pathways prior

In [4]:
hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list <- list(
  HALL     = hall_gmt,
  REACTOME = reactome_gmt,
  GOCC     = gocc_gmt,
  C8       = c8_gmt
)

all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, gtex_genes)
message("Loaded and matched all pathways matrix against GTEx genes")

There are 15980 genes in the intersection between data and prior

Removing 845 pathways

Loaded and matched all pathways matrix against GTEx genes



## Run CLAMPfull with all pathways prior

In [5]:
message("Running CLAMPfull with all pathways prior on GTEx (all samples)...")

gtex_fullRes_hall <- CLAMPfull(
  Y                 = gtex_fbm_filt,
  svdres            = gtex_svdRes,
  priorMat          = all_pathways_matched,
  clamp.base.result = gtex_baseRes,
  use_cpp           = TRUE,
  trace             = TRUE,
  multiplier        = MULTIPLIER,
  max.iter          = MAX_ITER,
  clamp_k           = CLAMP_K_gtex
)

gtex_fullRes_hall$Z <- data.frame(gtex_fullRes_hall$Z)
rownames(gtex_fullRes_hall$Z) <- gtex_genes

gtex_fullRes_hall$B <- data.frame(gtex_fullRes_hall$B)
colnames(gtex_fullRes_hall$B) <- samples

gtex_fullRes_hall$summary <- gtex_fullRes_hall$summary %>%
  dplyr::rename(LV = LV_index) %>%
  dplyr::mutate(LV = paste0('LV', LV))

message("CLAMPfull completed")

Running CLAMPfull with all pathways prior on GTEx (all samples)...

** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 578

L1=39.0160308826276; L2=117.048092647883

Progress 1 / 5000 | Bdiff=0.000440

Progress 2 / 5000 | Bdiff=0.037001

Progress 3 / 5000 | Bdiff=0.014585

Estimated total runtime: ~3608.4 min

Progress 4 / 5000 | Bdiff=0.009720

Progress 5 / 5000 | Bdiff=0.009923

Progress 6 / 5000 | Bdiff=0.007909

Progress 7 / 5000 | Bdiff=0.007080

Progress 8 / 5000 | Bdiff=0.006995

Progress 9 / 5000 | Bdiff=0.006507

Progress 10 / 5000 | Bdiff=0.006014

Progress 11 / 5000 | Bdiff=0.006247

Progress 12 / 5000 | Bdiff=0.006446

Progress 13 / 5000 | Bdiff=0.006344

Progress 14 / 5000 | Bdiff=0.006793

Progress 15 / 5000 | Bdiff=0.006624

Progress 16 / 5000 | Bdiff=0.006651

Progress 17 / 5000 | Bdiff=0.005969

Progress 18 / 5000 | Bdiff=0.005735

Progress 19 / 5000 | Bdiff=0.005861

Progress 20 / 5000 | Bdiff=0.005706

Progress 21 / 5000 | Bdiff=0.005541

Progress 22

## Save results

In [6]:
dst_dir <- file.path(output_write_dir, "CLAMPfull_hall")
dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

saveRDS(gtex_fullRes_hall, file = file.path(output_write_dir, "CLAMPfull_hall.rds"))

write.csv(gtex_fullRes_hall$B,       file.path(dst_dir, "B.csv"))
write.csv(gtex_fullRes_hall$Z,       file.path(dst_dir, "Z.csv"))
write.csv(gtex_fullRes_hall$summary, file.path(dst_dir, "summary.csv"))

message("Results saved to: ", dst_dir)

Results saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/02_gtex/10_CLAMP_hall/CLAMPfull_hall

